In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("hw7.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 7: Word embeddings and topic modeling 

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [2]:
import os

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline

<br><br>

<!-- BEGIN QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>


_Points:_ 2

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 1:  Exploring pre-trained word embeddings <a name="1"></a>
<hr>

In lecture 18, we talked about natural language processing (NLP). Using pre-trained word embeddings is very common in NLP. It has been shown that pre-trained word embeddings work well on a variety of text classification tasks. These embeddings are created by training a model like Word2Vec on a huge corpus of text such as a dump of Wikipedia or a dump of the web crawl. 

A number of pre-trained word embeddings are available out there. Some popular ones are: 

- [GloVe](https://nlp.stanford.edu/projects/glove/)
    * trained using [the GloVe algorithm](https://nlp.stanford.edu/pubs/glove.pdf) 
    * published by Stanford University 
- [fastText pre-trained embeddings for 294 languages](https://fasttext.cc/docs/en/pretrained-vectors.html) 
    * trained using the fastText algorithm
    * published by Facebook
    
In this exercise, you will be exploring GloVe Wikipedia pre-trained embeddings. The code below loads the word vectors trained on Wikipedia using an algorithm called Glove. You'll need `gensim` package in your cpsc330 conda environment to run the code below. 

```
> conda activate cpsc330
> conda install -c anaconda gensim
```

In [3]:
# !pip install gensim

import gensim
import gensim.downloader

print(list(gensim.downloader.info()["models"].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [4]:
# This will take a while to run when you run it for the first time.
import gensim.downloader as api

glove_wiki_vectors = api.load("glove-wiki-gigaword-100")

In [5]:
len(glove_wiki_vectors)

400000

There are 400,000 word vectors in this pre-trained model. 

Now that we have GloVe Wiki vectors loaded in `glove_wiki_vectors`, let's explore the embeddings. 

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

- Come up with a list of 4 words of your choice and find similar words to these words using `glove_wiki_vectors` embeddings.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 2

In [6]:
words_to_find = ["king", "queen", "city", "cat"]

for word in words_to_find:
    print(f"Most similar words to '{word}':")
    for similar_word in glove_wiki_vectors.most_similar(word, topn=5):
        print(similar_word)

Most similar words to 'king':
('prince', 0.7682328820228577)
('queen', 0.7507690787315369)
('son', 0.7020888328552246)
('brother', 0.6985775232315063)
('monarch', 0.6977890729904175)
Most similar words to 'queen':
('princess', 0.7947245240211487)
('king', 0.7507690191268921)
('elizabeth', 0.7355712056159973)
('royal', 0.7065026164054871)
('lady', 0.7044796943664551)
Most similar words to 'city':
('town', 0.8263899087905884)
('cities', 0.7764331698417664)
('where', 0.754779040813446)
('area', 0.7458435297012329)
('downtown', 0.7437540292739868)
Most similar words to 'cat':
('dog', 0.8798074722290039)
('rabbit', 0.7424427270889282)
('cats', 0.732300341129303)
('monkey', 0.7288709878921509)
('pet', 0.719014048576355)


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

1. Calculate cosine similarity for the following word pairs (`word_pairs`) using the [`similarity`](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) method of `glove_wiki_vectors`.

In [7]:
word_pairs = [
    ("coast", "shore"),
    ("clothes", "closet"),
    ("old", "new"),
    ("smart", "intelligent"),
    ("dog", "cat"),
    ("tree", "lawyer"),
]

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 2

In [8]:
for word1, word2 in word_pairs:
    similarity_score = glove_wiki_vectors.similarity(word1, word2)
    print(f"Similarity between '{word1}' and '{word2}': {similarity_score:.2f}")

Similarity between 'coast' and 'shore': 0.70
Similarity between 'clothes' and 'closet': 0.55
Similarity between 'old' and 'new': 0.64
Similarity between 'smart' and 'intelligent': 0.76
Similarity between 'dog' and 'cat': 0.88
Similarity between 'tree' and 'lawyer': 0.08


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Representation of all words in English
rubric={points}

**Your tasks:**

1. The vocabulary size of Wikipedia embeddings is quite large. The `test_words` list below contains a few new words (called neologisms) and biomedical domain-specific abbreviations. Write code to check whether `glove_wiki_vectors` have representation for these words or not. 
> If a given word `word` is in the vocabulary, `word in glove_wiki_vectors` will return True. 

In [9]:
test_words = [
    "covididiot",
    "fomo",
    "frenemies",
    "anthropause",
    "photobomb",
    "selfie",
    "pxg",  # Abbreviation for pseudoexfoliative glaucoma
    "pacg",  # Abbreviation for primary angle closure glaucoma
    "cct",  # Abbreviation for central corneal thickness
    "escc",  # Abbreviation for esophageal squamous cell carcinoma
]

<div class="alert alert-warning">

Solution_1_3
    
</div>

_Points:_ 2

In [10]:
for word in test_words:
    if word in glove_wiki_vectors:
        print(f"{word} is IN the model vocabulary.")
    else:
        print(f"{word} is -NOT- in the model vocabulary.")


covididiot is -NOT- in the model vocabulary.
fomo is -NOT- in the model vocabulary.
frenemies is IN the model vocabulary.
anthropause is -NOT- in the model vocabulary.
photobomb is -NOT- in the model vocabulary.
selfie is -NOT- in the model vocabulary.
pxg is -NOT- in the model vocabulary.
pacg is -NOT- in the model vocabulary.
cct is IN the model vocabulary.
escc is IN the model vocabulary.


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 Stereotypes and biases in embeddings
rubric={points}

Word vectors contain lots of useful information. But they also contain stereotypes and biases of the texts they were trained on. In the lecture, we saw an example of gender bias in Google News word embeddings. Here we are using pre-trained embeddings trained on Wikipedia data. 

**Your tasks:**

1. Explore whether there are any worrisome biases or stereotypes present in these embeddings by trying out at least 4 examples. You can use the following two methods or other methods of your choice to explore this. 
    - the `analogy` function below which gives word analogies (an example shown below)
    - [similarity](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) or [distance](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=distance#gensim.models.keyedvectors.KeyedVectors.distances) methods (an example is shown below)

> Note that most of the recent embeddings are de-biased. But you might still observe some biases in them. Also, not all stereotypes present in pre-trained embeddings are necessarily bad. But you should be aware of them when you use them in your models. 

In [11]:
def analogy(word1, word2, word3, model=glove_wiki_vectors):
    """
    Returns analogy word using the given model.

    Parameters
    --------------
    word1 : (str)
        word1 in the analogy relation
    word2 : (str)
        word2 in the analogy relation
    word3 : (str)
        word3 in the analogy relation
    model :
        word embedding model

    Returns
    ---------------
        pd.dataframe
    """
    print("%s : %s :: %s : ?" % (word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=["Analogy word", "Score"])

Examples of using analogy to explore biases and stereotypes.  

In [12]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


In [13]:
glove_wiki_vectors.similarity("aboriginal", "success")

np.float32(0.14283238)

In [14]:
glove_wiki_vectors.similarity("white", "success")

np.float32(0.35182396)

<div class="alert alert-warning">

Solution_1_4
    
</div>

_Points:_ 4

In [15]:
analogy("man", "engineer", "woman")

man : engineer :: woman : ?


,Analogy word,Score
0,technician,0.662020
1,educator,0.611515
2,engineers,0.592235
3,surgeon,0.588020
4,contractor,0.587330
5,engineering,0.568600
6,nurse,0.556342
7,chemist,0.554719
8,pioneer,0.546008
9,physician,0.545960


In [16]:
analogy("woman", "caring", "man")


woman : caring :: man : ?


,Analogy word,Score
0,loving,0.712168
1,nurturing,0.610965
2,sick,0.598257
3,compassionate,0.581296
4,hardworking,0.576437
5,good,0.572360
6,loved,0.570023
7,trusting,0.562382
8,life,0.559241
9,grateful,0.558070


In [17]:
analogy("man", "career", "woman")

man : career :: woman : ?


,Analogy word,Score
0,professional,0.621010
1,careers,0.616742
2,her,0.594655
3,she,0.585087
4,actress,0.582726
5,academic,0.576232
6,winning,0.574486
7,stint,0.573699
8,history,0.566176
9,years,0.553775


In [18]:
analogy("man", "intelligent", "woman")


man : intelligent :: woman : ?


,Analogy word,Score
0,articulate,0.627857
1,thoughtful,0.616820
2,smart,0.587221
3,strong-willed,0.568862
4,vivacious,0.550627
5,opinionated,0.547024
6,perceptive,0.545705
7,inquisitive,0.544539
8,energetic,0.533960
9,literate,0.533895


In [19]:
analogy("man", "intelligent", "man")

man : intelligent :: man : ?


,Analogy word,Score
0,smart,0.755273
1,thoughtful,0.710504
2,articulate,0.679080
3,sophisticated,0.666842
4,agile,0.665660
5,energetic,0.650417
6,clever,0.635688
7,resourceful,0.634564
8,incredibly,0.633014
9,passionate,0.622919


In [20]:
glove_wiki_vectors.similarity("man", "engineer")


np.float32(0.42998508)

In [21]:
glove_wiki_vectors.similarity("woman", "engineer")


np.float32(0.3340311)

In [22]:
analogy("man", "criminal", "woman")

man : criminal :: woman : ?


,Analogy word,Score
0,complaint,0.690229
1,prosecution,0.689820
2,crimes,0.687875
3,cases,0.677530
4,investigation,0.674370
5,charges,0.670625
6,harassment,0.658905
7,legal,0.651474
8,investigations,0.650149
9,crime,0.647365


In [23]:
glove_wiki_vectors.similarity("man", "criminal")

np.float32(0.4915791)

In [24]:
glove_wiki_vectors.similarity("woman", "criminal")


np.float32(0.39414236)

<!-- BEGIN QUESTION -->

### 1.5 Discussion
rubric={points}

**Your tasks:**
1. Discuss your observations from 1.4. Are there any worrisome biases in these embeddings trained on Wikipedia?   
2. Give an example of how using embeddings with biases could cause harm in the real world.

<div class="alert alert-warning">

Solution_1_5
    
</div>

_Points:_ 4

1.
analogy("man", "engineer", "woman") returned "technician", "educator". We can also observe that glove_wiki_vectors.similarity("man", "engineer") = 0.42, while glove_wiki_vectors.similarity("woman", "engineer") returned only 0.33. This reinforces the bias that only men are engineers and women are less associated with the word engineer, or worse degraded to "technician". 

glove_wiki_vectors.similarity("man", "criminal") = 0.49 while glove_wiki_vectors.similarity("woman", "criminal") = 0.39, this associates men more strongly with criminality than women.

It may be a worrisome bias depending on the downstream task these embeddings are to be used in. The embeddings could be attributed sexism in culture/societal stereotypes, or it could be explained that Wikipedia simply has many pages of historical figures, and engineers throughout history have been more men than women in engineering roles, leading to the embeddings associating the word engineer stronger with men than women. Similarly, it may be that more men have been criminals throughout history than women, leading to more wikipedia pages featuring male criminals.

2. 

Given that there is a bias in the learned embeddings, it would be wise to be careful how these embeddings are used in some machine learning or statistical modelling task. 

- Suppose that Amazon uses this to train their automated resume filtering model. It may inherit the biases within these embeddings, favouring men over women for engineering roles, reinforcing the gender stereotype in their hiring.
- Similarly, suppose that a government is using scanning a person's social media, emails, etc, as an early warning system for criminal behavior (disturbingly, this is not as sci-fi as one might think). Given that the model more strongly associates men with criminality than women, it may label men more readily as likely to commit a crime than women.


<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Topic modeling 

The goal of topic modeling is discovering high-level themes in a large collection of texts. 

In this homework, you will explore topics in [the 20 newsgroups text dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) using `scikit-learn`'s `LatentDirichletAllocation` (LDA) model. 

Usually, topic modeling is used for discovering abstract "topics" that occur in a collection of documents when you do not know the actual topics present in the documents. But 20 newsgroups text dataset is labeled with categories (e.g., sports, hardware, religion), and you will be able to cross-check the topics discovered by your model with these available topics. 

The starter code below loads the train and test portion of the data and convert the train portion into a pandas DataFrame. For speed, we will only consider documents with the following 8 categories. 

In [25]:
from sklearn.datasets import fetch_20newsgroups

In [26]:
cats = [
    "rec.sport.hockey",
    "rec.sport.baseball",
    "soc.religion.christian",
    "alt.atheism",
    "comp.graphics",
    "comp.windows.x",
    "talk.politics.mideast",
    "talk.politics.guns",
]  # We'll only consider these categories out of 20 categories for speed.

newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes"), categories=cats
)
X_news_train, y_news_train = newsgroups_train.data, newsgroups_train.target
df = pd.DataFrame(X_news_train, columns=["text"])
df["target"] = y_news_train
df["target_name"] = [
    newsgroups_train.target_names[target] for target in newsgroups_train.target
]
df

,text,target,target_name
0,"You know, I was reading 18 U.S.C. 922 and some...",6,talk.politics.guns
1,\n\n\nIt's not a bad question: I don't have an...,1,comp.graphics
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics
3,"The following problem is really bugging me,\na...",2,comp.windows.x
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast
...,...,...,...
4558,Hi Everyone ::\n\nI am looking for some soft...,1,comp.graphics
4559,Archive-name: x-faq/part3\nLast-modified: 1993...,2,comp.windows.x
4560,"\nThat's nice, but it doesn't answer the quest...",6,talk.politics.guns
4561,"Hi,\n I just got myself a Gateway 4DX-33V ...",2,comp.windows.x


In [27]:
newsgroups_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.windows.x',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast']

<br><br>

<!-- BEGIN QUESTION -->

### 2.1 Preprocessing using [spaCy](https://spacy.io/)
rubric={points}

Preprocessing is a crucial step before carrying out topic modeling and it markedly affects topic modeling results. In this exercise, you'll prepare the data using [spaCy](https://spacy.io/) for topic modeling. 

**Your tasks:** 

- Write code using [spaCy](https://spacy.io/) to preprocess the `text` column in the given dataframe `df` and save the processed text in a new column called `text_pp` within the same dataframe.

If you do not have spaCy in your course environment, you'll have to [install it](https://spacy.io/usage) and download the pretrained model en_core_web_md. 

`python -m spacy download en_core_web_md`


Note that there is no such thing as "perfect" preprocessing. You'll have to make your own judgments and decisions on which tokens are likely to be more informative for the given task. Some common text preprocessing steps for topic modeling include: 
- getting rid of slashes, new-line characters, or any other non-informative characters
- sentence segmentation and tokenization      
- replacing urls, email addresses, or numbers with generic tokens such as "URL",  "EMAIL", "NUM". 
- getting rid of other fairly unique tokens which are not going to help us in topic modeling  
- excluding stopwords and punctuation 
- lemmatization


> Check out [these available attributes](https://spacy.io/api/token#attributes) for `token` in spaCy which might help you with preprocessing. 

> You can also get rid of words with specific POS tags. [Here](https://universaldependencies.org/u/pos/) is the list of part-of-speech tags used in spaCy. 

> You may have to use regex to clean text before passing it to spaCy. Also, you might have to go back and forth between preprocessing in this exercise and and topic modeling in Exercise 2 before finalizing preprocessing steps. 

> Note that preprocessing the corpus might take some time. So here are a couple of suggestions: 1) During the debugging phase, work on a smaller subset of the data. 2) Once you finalize the preprocessing part, you might want to save the preprocessed data in a CSV and work with this CSV so that you don't run the preprocessing part every time you run the notebook. 
 


In [28]:
# !pip install spacy
# !python -m spacy download en_core_web_md

import spacy
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

<div class="alert alert-warning">

Solution_2_1
    
</div>

_Points:_ 8

In [29]:
# regular expressions were implemented with the aid of ChatGPT
# prompt: generate regular expressions for preprocessing text data replacing urls, email addresses, or numbers with generic tokens such as "URL",  "EMAIL", "NUM". 

import re

def preprocess(str):
    str = str.replace("/", " ")
    str = str.replace("\\", " ")
    str = str.replace("\n", " ")

    str = re.sub(r"https?://\S+|www\.\S+", " URL ", str)
    str = re.sub(r"\S+@\S+", " EMAIL ", str)
    str = re.sub(r"\b\d+(\.\d+)?\b", " NUM ", str)

    doc = nlp(str)

    processed_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        lemma = token.lemma_
        processed_tokens.append(lemma)


    return " ".join(processed_tokens)

    

In [30]:
df.iloc[2:6]

,text,target,target_name
2,"\nActuallay I don't, but on the other hand I d...",1,comp.graphics
3,"The following problem is really bugging me,\na...",2,comp.windows.x
4,\n\n This is the latest from UPI \n\n For...,7,talk.politics.mideast
5,"Hi,\n I'd like to subscribe to Leadership Ma...",5,soc.religion.christian


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Build a topic model using sklearn's LatentDirichletAllocation
rubric={points}

**Your tasks:**

1. Build LDA models on the preprocessed data using using [sklearn's `LatentDirichletAllocation`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) and random state 42. Experiment with a few values for the number of topics (`n_components`). Pick a reasonable number for the number of topics and briefly justify your choice.

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Points:_ 4

_Type your answer here, replacing this text._

In [31]:
df["processed_text"] = df["text"].apply(preprocess)

In [32]:
df["processed_text"].iloc[2:6]

2      Actuallay hand support idea have newsgroup a...
3    follow problem bug appreciate help   create wi...
4         late UPI         Foreign Ministry spokesm...
5    hi     like subscribe Leadership Magazine wond...
Name: processed_text, dtype: object

In [33]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

df["target_name"].nunique() # 8
print(df["target_name"].unique())

num_topics_adjustment = -1

vectorizer = CountVectorizer();
X = vectorizer.fit_transform(df["processed_text"]);

lda = LatentDirichletAllocation(n_components=df["target_name"].nunique() + num_topics_adjustment, random_state=42);
lda.fit(X);

['talk.politics.guns' 'comp.graphics' 'comp.windows.x'
 'talk.politics.mideast' 'soc.religion.christian' 'rec.sport.hockey'
 'alt.atheism' 'rec.sport.baseball']


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.3 Exploring word topic association
rubric={points}

**Your tasks:**
1. For the number of topics you picked in the previous exercise, show top 10 words for each of your topics and suggest labels for each of the topics (similar to how we came up with labels "health and nutrition", "fashion", and "machine learning" in the toy example we saw in class). 

> If your topics do not make much sense, you might have to go back to preprocessing in Exercise 2.1, improve it, and train your LDA model again. 

<div class="alert alert-warning">

Solution_2_3
    
</div>

_Points:_ 5

In [37]:
feature_names = vectorizer.get_feature_names_out()

def get_top_words_per_topic(model, feature_names, n_top_words):
    topics = []

    for topic in model.components_:
        sorted_indices = topic.argsort()

        top_words = []
        count = 0

        for i in range(len(sorted_indices) - 1, -1, -1):
            idx = sorted_indices[i]
            top_words.append(feature_names[idx])
            count += 1

            if count == n_top_words:
                break

        topics.append(top_words)

    return topics

topics = get_top_words_per_topic(lda, feature_names, 10)

for i in range(len(topics)):
    print("Topic", i, ":", topics[i])

Topic 0 : ['num', 'email', 'play', 'period', 'game', 'team', 'la', 'new', 'pt', 'win']
Topic 1 : ['num', 'file', 'email', 'program', 'image', 'use', 'include', 'available', 'entry', 'version']
Topic 2 : ['window', 'thank', 'work', 'key', 'motif', 'problem', 'like', 'xterm', 'use', 'application']
Topic 3 : ['num', 'people', 'armenian', 'israel', 'turkish', 'jews', 'armenians', 'israeli', 'government', 'war']
Topic 4 : ['gun', 'people', 'weapon', 'law', 'num', 'firearm', 'state', 'right', 'crime', 'control']
Topic 5 : ['num', 'say', 'know', 'year', 'think', 'go', 'game', 'good', 'come', 'time']
Topic 6 : ['god', 'people', 'num', 'believe', 'think', 'know', 'jesus', 'say', 'question', 'thing']


<!-- END QUESTION -->

<br><br>

0: sports
1: computer software
2: Linux
3: world news
4: firearms policy
5: general news
6: religion

<!-- BEGIN QUESTION -->

### 2.4 Exploring document topic association
rubric={points}

**Your tasks:**
1. Show the document topic assignment of the first five documents from `df`.

<div class="alert alert-warning">

Solution_2_4
    
</div>

_Points:_ 5

In [40]:
doc_topic_dist = lda.transform(X)
doc_topics = np.argmax(doc_topic_dist, axis=1)

for i in range(5):
    print("Document", i)
    print("Assigned topic:", doc_topics[i])
    print("Topic distribution:", doc_topic_dist[i])
    print()

Document 0
Assigned topic: 4
Topic distribution: [0.10478383 0.00224324 0.00224228 0.00224257 0.60652988 0.08905247
 0.19290573]

Document 1
Assigned topic: 1
Topic distribution: [0.00184033 0.54038187 0.00183778 0.33194922 0.00183996 0.12031144
 0.00183939]

Document 2
Assigned topic: 6
Topic distribution: [0.00230858 0.34124785 0.00231863 0.1227855  0.00231206 0.08824585
 0.44078153]

Document 3
Assigned topic: 2
Topic distribution: [0.00397534 0.00397658 0.6335693  0.34654487 0.00397624 0.00397332
 0.00398435]

Document 4
Assigned topic: 3
Topic distribution: [0.00317619 0.00318472 0.23873501 0.74534809 0.00318557 0.00318616
 0.00318426]



<!-- BEGIN QUESTION -->

## Exercise 3: Short answer questions 
<hr>

rubric={points}

1. Briefly explain how content-based filtering works in the context of recommender systems. 
2. Discuss at least two negative consequences of recommender systems.
3. What is transfer learning in natural language processing? Briefly explain.     

<div class="alert alert-warning">

Solution_3
    
</div>

_Points:_ 6

1. 
Content based filtering works by taking the metadata of, for example movies or books, as features (ie. genre, year, etc); or more sophisticatedly, text features or plot summaries or learned embeddings may be used instead of these simple features. Then from these features, each person's ratings of movies are used to build a weight vector of how much they value each feature. Then new movies may be recommended based on how much they value each feature.

2. 
- Cold start: with collaborative filtering, when there is a new item, a movie for example, and there are little to no users rating that item, the system struggles to identify its latent features due to a lack of data. 
- Recommending new things: with content based recommendation (and may apply to other recommender systems), an individual's weight vector denotes the kinds of features they may like (such as genre, etc). The recommender may get stuck on recommending only things that are very much like the previous items that individual liked, resulting in little exploration into new unseen potential interests.

3. 
Transfer learning (in the context of NLP) is taking a pretrained model already trained on a (large) body of text and using that for the basis to train a more specialized task. 

<!-- END QUESTION -->

<br><br><br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)